In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '1'
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"
from functools import partial
import time
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
from copy import deepcopy

import matplotlib as mpl
from matplotlib import rc
rc('font',**{'family':'serif','serif':['Helvetica']})
mpl.rcParams['text.usetex'] = True
mpl.rcParams.update({'font.size': 10})
mpl.rcParams['text.latex.preamble']=r"\usepackage{bm}\usepackage{amsmath}\usepackage{upgreek}"

In [ ]:
import jax
import jax.numpy as jnp
# jax.config.update("jax_enable_x64", True)
# jax.config.update("jax_debug_nans", True)
gpus = jax.devices()
print(gpus)

jax.config.update("jax_default_device", gpus[0])

import diffrax
import equinox as eqx
import optax

from haiku import PRNGSequence

from dmpe.data_management import DataPaths
from dmpe.evaluation.plotting_utils import plot_sequence
from dmpe.evaluation.experiment_utils import get_experiment_ids, load_experiment_results
from dmpe.models.models import NeuralEulerODECartpole
from dmpe.models.model_utils import simulate_ahead_with_env

In [ ]:
from dmpe.utils.env_utils.fluid_tank_utils import setup_env as setup_fluid_tank_env
from dmpe.utils.env_utils.pendulum_utils import setup_env as setup_pendulum_env
from dmpe.utils.env_utils.cart_pole_utils import setup_env as setup_cart_pole_env

In [ ]:
from dmpe.utils.density_estimation import build_grid
from dmpe.models.model_utils import simulate_ahead_with_env

In [ ]:
from dmpe.utils.sets.reachable_set import approximate_reachable_set
from dmpe.utils.sets.control_invariant_set import approximate_control_invariant_set

from dmpe.utils.sets.reachable_set import save_results as save_results_rs
from dmpe.utils.sets.control_invariant_set import save_results as save_results_ci
from dmpe.utils.sets.shared import load_results, DiscretizedSet, SlicedSet, load_discretized_set, save_discretized_set

In [ ]:
from enum import Enum
class Systems(Enum):
    FLUID_TANK = 1
    PENDULUM = 2
    CART_POLE = 3

In [ ]:
sys_name = Systems.PENDULUM

if sys_name == Systems.FLUID_TANK:
    env, penalty_function, featurize, _ = setup_fluid_tank_env()
    R_s = load_discretized_set(DataPaths().reach_ci_experiments / "fluid_tank_Rs_d4557af7-1977-43.json")
    C = load_discretized_set(DataPaths().reach_ci_experiments / "fluid_tank_C_93ce41af-dd22-4e.json")
elif sys_name == Systems.PENDULUM:
    env, penalty_function, featurize, _ = setup_pendulum_env()
    R_s = load_discretized_set(DataPaths().reach_ci_experiments / "pendulum_Rs_3ee859d2-42fd-42.json")
    C = load_discretized_set(DataPaths().reach_ci_experiments / "pendulum_C_18209bdb-58e2-43.json")
elif sys_name == Systems.CART_POLE:
    env, penalty_function, featurize, _ = setup_cart_pole_env()
    R_s = load_discretized_set(DataPaths().reach_ci_experiments / "cart_pole_Rs_a10d3279-68f6-4d.json")
    C = load_discretized_set(DataPaths().reach_ci_experiments / "cart_pole_C_069e72af-6e6f-46.json")

R_s.visualize()
plt.show()
C.visualize()
plt.show()

In [ ]:
R_s.visualize(reduction_method=jnp.sum, labels=env.obs_description, use_contourf=False)
plt.show()

C.visualize(reduction_method=jnp.sum, labels=env.obs_description, use_contourf=False)
plt.show()

In [ ]:
S = R_s & C
S.visualize(reduction_method=jnp.any, labels=env.obs_description, use_contourf=False)
plt.show()

S.visualize(reduction_method=jnp.sum, labels=env.obs_description, use_contourf=False)
plt.show()

# C(x) to C(x, u)

In [ ]:
C.visualize(jnp.sum, use_contourf=True)

In [ ]:
observations = C.grid

if sys_name == Systems.CART_POLE:
    actions = jnp.linspace(-1, 1, 25)[..., None]
else:
    actions = jnp.linspace(-1, 1, 50)[..., None]

In [ ]:
def fuse_obs_act(observations, action):
    return jnp.concatenate([observations, action.repeat(observations.shape[0])[:, None]], axis=-1)

In [ ]:
@eqx.filter_jit
def step_from_init_obs(init_obs, action, env):
    init_state = env.generate_state_from_observation(init_obs, env.env_properties)
    next_obs, _ = env.step(init_state, action, env.env_properties)
    return next_obs

In [ ]:
obs_dim = env.reset(env.env_properties)[0].shape[-1]

fused = eqx.filter_vmap(fuse_obs_act, in_axes=(None, 0), out_axes=(1))(observations, actions).reshape(-1, obs_dim+env.action_dim)

# simulate one step ahead
next_observations = eqx.filter_vmap(eqx.filter_vmap(step_from_init_obs, in_axes=(None, 0, None)), in_axes=(0, None, None))(observations, actions, env)

# next_observations.reshape([points_per_dim] * dim + [-1]).shape
next_observations = next_observations.reshape([-1, obs_dim])

In [ ]:
out = []
chunk_size = 20_000
n_observations = next_observations.shape[0]

for i in tqdm(jnp.arange(0, n_observations, chunk_size)):
    next_obs = next_observations[i : min(i + chunk_size, n_observations)] 
    out.append(eqx.filter_vmap(C.check_in_set)(next_obs))
out = jnp.concatenate(out)

In [ ]:
C_xu = DiscretizedSet(
    grid=fused,
    mask=out,
    unflattened_shape=tuple([*C.unflattened_shape, actions.shape[0]]),
)

In [ ]:
C_xu

In [ ]:
C_xu.visualize(jnp.sum, labels=[*env.obs_description, *env.action_description])

In [ ]:
R_s_xu = DiscretizedSet(
    grid=C_xu.grid,
    mask=R_s.mask[..., None].repeat(C_xu.unflattened_shape[-1], axis=-1).flatten(),
    unflattened_shape=C_xu.unflattened_shape,
)

In [ ]:
S_xu = C_xu & R_s_xu

In [ ]:
S_xu.visualize()

### store results:

In [ ]:
from uuid import uuid4

In [ ]:
exp_id = str(uuid4())[:16]
save_discretized_set(
    DataPaths().reach_ci_experiments / f"{str(sys_name.name).lower()}_S_xu_{exp_id}.json",
    S_xu
)

In [ ]:
test = load_discretized_set(
    DataPaths().reach_ci_experiments / f"{str(sys_name.name).lower()}_S_xu_{exp_id}.json",
)
test.visualize()